# Contract Compliance & Invoice Verification

A real-world accounts-payable automation demo: verify a consulting invoice against two signed contracts, flag discrepancies, and simulate what-if payment scenarios.

This notebook stresses litesearch's full feature set:

1. **Hierarchical chunking** — Payment Terms section (parent) + rate/milestone table rows (children)
2. **Multimodal table extraction** — `pdf_images_with_captions()` captures rate-card tables even when PDF text extraction is imperfect
3. **Cross-document RAG** — contract terms + invoice data retrieved in a single query
4. **Custom Lisette tool** — `query_invoice_db` gives the LLM structured SQL access to invoice line items
5. **Agentic reasoning** — `LiteRAG` + a cloud LLM handles arithmetic, citations, and what-if scenarios

**Source documents** (all public, no login required):
- [`burnaby_consulting.pdf`](https://www.burnaby.ca/sites/default/files/acquiadam/2021-07/Standard-Consulting-Agreement-Deliverables-Payment-Schedule-A.pdf) — ARTICLE III payment terms + Schedule A milestone table
- [`master_consulting.pdf`](https://tcfoe.com/wp-content/uploads/2023/03/Master-Consulting-Agreement-SAMPLE.pdf) — Exhibit A rate card ($200/hr), expense caps, net-30 rules
- [`invoice_sample.pdf`](https://slicedinvoices.com/pdf/wordpress-pdf-invoice-plugin-sample.pdf) — Sample consulting invoice with line items

> Requires `pip install litesearch lisette chonkie[all]` and a cloud API key (Claude or OpenAI).
> Reranking requires `pip install flashrank`.

## 1. Setup

In [ ]:
import os, json, urllib.request
from pathlib import Path
from litesearch import *
from litesearch.data import hierarchical_pdf_texts, pdf_images_with_captions
from litesearch.rag import LiteRAG
from litesearch.utils import FastRerank

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

ENCODER_MODEL = 'minishlab/potion-retrieval-32M'

# Cloud LLM recommended for arithmetic and multi-document reasoning
# Alternatives: 'openai/gpt-4o-mini', 'openai/gpt-4o', 'claude-sonnet-4-6'
LLM_MODEL = 'claude-haiku-4-5-20251001'

DB_PATH = 'contract_compliance.db'

## 2. Download PDFs

In [ ]:
pdf_dir = Path('pdfs_contract_demo')
pdf_dir.mkdir(exist_ok=True)

pdfs = {
    'burnaby_consulting.pdf': 'https://www.burnaby.ca/sites/default/files/acquiadam/2021-07/Standard-Consulting-Agreement-Deliverables-Payment-Schedule-A.pdf',
    'master_consulting.pdf':  'https://tcfoe.com/wp-content/uploads/2023/03/Master-Consulting-Agreement-SAMPLE.pdf',
    'invoice_sample.pdf':     'https://slicedinvoices.com/pdf/wordpress-pdf-invoice-plugin-sample.pdf',
}

# Some servers reject Python's default UA — use a browser UA
headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36'}

for name, url in pdfs.items():
    dest = pdf_dir / name
    if dest.exists():
        print(f'{name} already present ({dest.stat().st_size:,} bytes)')
        continue
    print(f'Downloading {name}...')
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as resp, open(dest, 'wb') as f:
        f.write(resp.read())
    print(f'  → saved {dest.stat().st_size:,} bytes')

# Verify with pdf_oxide
from pdf_oxide import PdfDocument
for name in pdfs:
    doc = PdfDocument(str(pdf_dir / name))
    print(f'{name}: {doc.page_count()} pages')

## 3. Ingest Contracts — Hierarchical Text Chunks

Each contract is split into **parent chunks** (512 tokens, e.g. an entire "Payment Terms" clause) and **child chunks** (128 tokens, e.g. one rate-table row). Hybrid FTS5+vector search operates on children; `expand_parents=True` in `LiteRAG` fetches the parent for full context when generating answers.

In [ ]:
db = database(DB_PATH)
store = db.get_store()
print('Store columns:', [c.name for c in store.columns])

In [ ]:
from chonkie import AutoEmbeddings
_enc = AutoEmbeddings().get_embeddings(ENCODER_MODEL)
encoder = lambda texts: _enc.embed(texts)

In [ ]:
contract_pdfs = ['burnaby_consulting.pdf', 'master_consulting.pdf']

if len(store()) == 0:
    for name in contract_pdfs:
        path = str(pdf_dir / name)
        print(f'Ingesting {name}...')
        chunks = hierarchical_pdf_texts(path, use_parent_child=True,
                                        parent_size=512, child_size=128)
        store.ingest_chunks(chunks, encoder=encoder)
        parents  = sum(1 for c in chunks if c.get('hierarchy_level') == 1)
        children = sum(1 for c in chunks if c.get('hierarchy_level') == 2)
        print(f'  → {parents} parents, {children} children')

print(f'Total text records: {len(store())}')

## 4. Extract Rate-Card Tables via Image Captions

Rate schedules and milestone tables in PDFs often render as images or complex layouts that text extractors mangle. `pdf_images_with_captions()` extracts each image paired with its figure/table caption (e.g. *"Table 1: Consultant Rate Schedule"*) so that cross-modal RRF can retrieve them semantically even when the tabular text is garbled.

In [ ]:
existing_img = sum(1 for r in store() if r.get('chunk_type') == 'image')

if existing_img == 0:
    for name in contract_pdfs:
        path = str(pdf_dir / name)
        print(f'Extracting tables/figures: {name}...')
        img_records = pdf_images_with_captions(path)
        if img_records:
            embs = encoder([r['content'] for r in img_records])
            for r, e in zip(img_records, embs):
                r['embedding'] = e.tobytes()
            store.insert_all(img_records)
            print(f'  → {len(img_records)} image/table records')
            for r in img_records:
                print(f'    caption: {r["content"][:80]}')
        else:
            print(f'  → no captioned images found')

total_img = sum(1 for r in store() if r.get('chunk_type') == 'image')
print(f'Total image records: {total_img}')

## 5. Ingest Invoice PDF + Build Structured Invoice Table

The sample invoice PDF is ingested as text chunks for semantic search. In addition, we build a **structured SQLite table** (`invoices`) with line items matching the contract rate card — but with **3 deliberate discrepancies** for the LLM to detect:

| Line | Item | Issue |
|------|------|-------|
| 4 | Inspection Services | Rate billed at **$250/hr** vs contract rate of **$200/hr** |
| 5 | Stakeholder Interviews | Math error: 10 hrs × $200 = **$2,000** but invoice shows **$2,200** |
| 6 | Accommodation & Per Diem | Amount **$980** exceeds the **5% disbursement cap** |


In [ ]:
# Ingest invoice PDF as flat text chunks (no parent/child needed for a short invoice)
inv_path = str(pdf_dir / 'invoice_sample.pdf')
inv_chunks = hierarchical_pdf_texts(inv_path, use_parent_child=False)
if inv_chunks:
    store.ingest_chunks(inv_chunks, encoder=encoder)
    print(f'Ingested {len(inv_chunks)} invoice text chunks')

In [ ]:
# Structured invoice table — rates derived from master_consulting.pdf Exhibit A ($200/hr)
# Burnaby contract Article III: disbursements capped at 5% of professional fees
invoice_rows = [
    # --- Correct line items ---
    {'line': 1, 'description': 'Workshop Facilitation',    'hours': 3,    'rate': 200.00, 'amount': 600.00,  'status': 'OK',    'note': 'Rate and math correct'},
    {'line': 2, 'description': 'Project Management',       'hours': 8,    'rate': 200.00, 'amount': 1600.00, 'status': 'OK',    'note': 'Rate and math correct'},
    {'line': 3, 'description': 'Travel & Disbursements',   'hours': None, 'rate': None,   'amount': 110.00,  'status': 'OK',    'note': 'Within 5% of fees ($2,200 × 5% = $110)'},
    # --- Discrepant line items ---
    {'line': 4, 'description': 'Inspection Services',      'hours': 5,    'rate': 250.00, 'amount': 1250.00, 'status': 'ERROR', 'note': 'Rate $250/hr exceeds contract rate $200/hr'},
    {'line': 5, 'description': 'Stakeholder Interviews',   'hours': 10,   'rate': 200.00, 'amount': 2200.00, 'status': 'ERROR', 'note': 'Math error: 10 × $200 = $2,000, not $2,200'},
    {'line': 6, 'description': 'Accommodation & Per Diem', 'hours': None, 'rate': None,   'amount': 980.00,  'status': 'ERROR', 'note': 'Exceeds 5% disbursement cap (~$190 max on $3,800 fees)'},
]

db['invoices'].insert_all(invoice_rows, ignore=True)
print(f'Invoice table created: {len(list(db["invoices"].rows))} rows')
for row in db['invoices'].rows:
    hrs   = f"{row['hours']}h" if row['hours'] else 'flat'
    rate  = f"@${row['rate']:.2f}" if row['rate'] else ''
    print(f"  Line {row['line']}: {row['description']:<30} {hrs} {rate} = ${row['amount']:.2f}  [{row['status']}]")

## 6. Define Invoice Tool + Create LiteRAG

`query_invoice_db` is a plain Python function (no `@tool` decorator) that Lisette automatically wraps into a tool. The LLM can call it with any `SELECT` query to retrieve structured invoice data alongside RAG context from the contracts.

In [ ]:
def query_invoice_db(sql: str) -> str:
    """Execute a SELECT query against the invoices table and return results as plain text.
    Table schema: line (int), description (text), hours (real), rate (real), amount (real), status (text), note (text).
    Example: SELECT * FROM invoices WHERE status='ERROR'"""
    try:
        rows = list(db.q(sql))
        if not rows:
            return 'No results.'
        return '\n'.join(str(dict(r)) for r in rows)
    except Exception as e:
        return f'[SQL error: {e}]'

In [ ]:
rag = LiteRAG(
    db_path=DB_PATH,
    encoder_model=ENCODER_MODEL,
    llm_model=LLM_MODEL,
    expand_parents=True,          # fetch full Payment Terms section, not just the rate row
    k=10,                         # candidates before rerank
    top_n=5,                      # final passages to LLM
    reranker=FastRerank(model='ms-marco-MiniLM-L-12-v2', max_length=160),
    tools=[query_invoice_db],
    search_level=None,            # no web search for offline contract queries
)
print('LiteRAG ready:', rag.llm_model)

## 7. Compliance Queries

### Query 1 — Rate Extraction + Arithmetic

LiteRAG retrieves the Exhibit A rate card from the Master Consulting Agreement (child chunk hit → parent context expansion), then the LLM queries the invoice table and checks each line.

In [ ]:
response = rag.query(
    'Extract the applicable hourly rate from the Master Consulting Agreement rate table. '
    'Then query the invoice database and apply that rate to each hourly line item. '
    'Show your math for each line and flag any discrepancies.',
    max_steps=5
)
print(response.choices[0].message.content)

### Query 2 — Milestone & Payment Term Compliance

Tests hierarchical retrieval: the Payment Terms clause (parent) contains the disbursement cap rule; the Schedule A table (child) has individual milestone percentages.

In [ ]:
response = rag.query(
    'According to the Payment Terms section and Schedule A milestone table in the Burnaby '
    'Standard Consulting Agreement, what is the maximum allowable disbursement percentage? '
    'Check whether the Travel & Disbursements and Accommodation & Per Diem lines in the invoice comply.',
    max_steps=5
)
print(response.choices[0].message.content)

### Query 3 — Cross-Document Table Comparison

Cross-modal search: rate card from the master contract (possibly extracted as an image caption) vs. the invoice table. Demonstrates RRF merging text and image-caption hits.

In [ ]:
response = rag.query(
    'Describe the services and rates table in the Master Consulting Agreement and compare it '
    'to every line item in the invoice. For each invoice line, state whether the rate is '
    'consistent with the contract, citing the relevant contract section or table.',
    max_steps=5
)
print(response.choices[0].message.content)

### Query 4 — What-If Scenario Reasoning

Agentic multi-step: calculate corrected totals, then simulate two alternative scenarios.

In [ ]:
response = rag.query(
    'Calculate the correct total due on this invoice after applying the contracted $200/hr rate '
    'and fixing any arithmetic errors. Then simulate two what-if scenarios: '
    '(A) apply a spot rate of $250/hr only to the Inspection Services line as a one-time exception; '
    '(B) add 2% monthly late interest on the entire corrected invoice if payment is 30 days overdue. '
    'Show the final payable amount for each scenario.',
    max_steps=6
)
print(response.choices[0].message.content)

### Query 5 — Full Compliance Report

Synthesises both contracts and the invoice. Uses `search_level='m'` to allow a web search fallback for Canadian consulting billing standards if the local documents are insufficient.

In [ ]:
# Rebuild LiteRAG with web search enabled for this final query
rag_with_search = LiteRAG(
    db_path=DB_PATH,
    encoder_model=ENCODER_MODEL,
    llm_model=LLM_MODEL,
    expand_parents=True,
    k=10,
    top_n=5,
    reranker=FastRerank(model='ms-marco-MiniLM-L-12-v2', max_length=160),
    tools=[query_invoice_db],
    search_level='m',   # enables web search fallback
)

response = rag_with_search.query(
    'Review the invoice against both the Burnaby Standard Consulting Agreement and the '
    'Master Consulting Agreement. Produce a structured compliance report that: '
    '(1) lists every line item with a PASS/FAIL verdict and the reason; '
    '(2) shows the corrected amount for each failed line; '
    '(3) states the corrected invoice total; '
    '(4) cites the specific contract clause or section for each finding. '
    'Reference standard Canadian consulting billing practices where applicable.',
    max_steps=8
)
print(response.choices[0].message.content)

## 8. Compliance Summary Table

Pull the expected vs. actual amounts directly from the invoice database and display a quick summary.

In [ ]:
rows = list(db['invoices'].rows)

def corrected_amount(row):
    """Apply contract rules to compute the correct amount."""
    if row['line'] == 4:   # Rate should be $200, not $250
        return row['hours'] * 200.00
    if row['line'] == 5:   # Fix arithmetic
        return row['hours'] * row['rate']
    if row['line'] == 6:   # Cap at 5% of professional fees (Lines 1+2+4+5 corrected)
        prof_fees = 600 + 1600 + 1000 + 2000  # corrected lines 1,2,4,5
        return round(prof_fees * 0.05, 2)
    return row['amount']   # already correct

header = f"{'Line':<5} {'Description':<30} {'Billed':>10} {'Correct':>10} {'Status':<8} Note"
print(header)
print('-' * len(header))

total_billed = 0
total_correct = 0
for row in rows:
    billed  = row['amount']
    correct = corrected_amount(row)
    total_billed  += billed
    total_correct += correct
    flag = '✓' if row['status'] == 'OK' else '✗'
    print(f"{row['line']:<5} {row['description']:<30} {billed:>10.2f} {correct:>10.2f} {flag:<8} {row['note']}")

print('-' * len(header))
print(f"{'TOTAL':<36} {total_billed:>10.2f} {total_correct:>10.2f}")
print(f"\nOvercharge detected: ${total_billed - total_correct:,.2f}")

## Cleanup

In [ ]:
# Optionally remove the demo database and downloaded PDFs
# import shutil
# os.remove(DB_PATH)
# shutil.rmtree(pdf_dir)